# Phase 1 — Weekly Data Pipeline

Inputs (copy to `data/raw/`):
- `Year_2009-2010_post.parquet`
- `Year_2010-2011_post.parquet`

Outputs:
- `data/weekly_train.parquet`
- `data/weekly_val.parquet`
- `data/weekly_test.parquet`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../data')
OUT.mkdir(exist_ok=True)

## 1. Load & merge raw parquets

In [2]:
df = pd.concat([
    pd.read_parquet(RAW / 'Year_2009-2010_post.parquet'),
    pd.read_parquet(RAW / 'Year_2010-2011_post.parquet'),
], ignore_index=True)

print(f'Raw rows: {len(df):,}')
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'Countries: {df["Country"].value_counts().head()}')

Raw rows: 1,067,008
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Countries: Country
United Kingdom    980967
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Name: count, dtype: int64


## 2. Filter: UK only, drop returns and zero-price rows

In [3]:
df = df[df['Country'] == 'United Kingdom'].copy()
df = df[df['Quantity'] > 0]       # drop returns (negative quantity)
df = df[df['Price'] > 0]          # drop zero-price rows

print(f'After UK + quality filter: {len(df):,} rows')

After UK + quality filter: 958,502 rows


## 3. Compute target and weekly aggregation

In [4]:
df['total_sales'] = df['Quantity'] * df['Price']

# ISO week Monday as week_start
df['week_start'] = df['InvoiceDate'].dt.to_period('W-MON').dt.start_time

weekly = (
    df.groupby(['product_family_name', 'week_start'], as_index=False)
    .agg(weekly_sales=('total_sales', 'sum'))
)

print(f'Weekly rows: {len(weekly):,}')
print(f'Unique product families: {weekly["product_family_name"].nunique()}')
print(f'Week range: {weekly["week_start"].min()} → {weekly["week_start"].max()}')
weekly.head()

Weekly rows: 113,282
Unique product families: 2574
Week range: 2009-12-01 00:00:00 → 2011-12-06 00:00:00


,product_family_name,week_start,weekly_sales
0,*Boombox Ipod Classic,2010-12-07,33.96
1,*USB Office Glitter Lamp,2009-12-01,8.65
2,*USB Office Mirror Ball,2009-12-01,8.65
3,*USB Office Mirror Ball,2010-12-14,16.94
4,10 COLOUR SPACEBOY PEN,2010-05-04,226.08


## 4. Build full weekly grid (fill missing weeks with 0)

In [5]:
# Derive all_weeks from the sparse data to avoid Timestamp precision mismatch with pd.date_range
# (pd.date_range freq='W-MON' produces different Timestamp objects than dt.to_period().dt.start_time,
#  causing a silent merge failure where all weekly_sales become NaN → 0)
all_weeks = sorted(weekly['week_start'].unique())
all_products = weekly['product_family_name'].unique()

grid = pd.MultiIndex.from_product(
    [all_products, all_weeks],
    names=['product_family_name', 'week_start']
).to_frame(index=False)

weekly = grid.merge(weekly, on=['product_family_name', 'week_start'], how='left')
weekly['weekly_sales'] = weekly['weekly_sales'].fillna(0.0)

print(f'Grid rows: {len(weekly):,}  (products × weeks = {len(all_products)} × {len(all_weeks)})')
print(f'Non-zero rows: {(weekly["weekly_sales"] > 0).sum():,}')
print(f'Max weekly_sales: {weekly["weekly_sales"].max():.2f}')

Grid rows: 270,270  (products × weeks = 2574 × 105)
Non-zero rows: 113,282
Max weekly_sales: 168469.60


## 5. Add UK holiday features

In [6]:
# pip install workalendar
from workalendar.europe import UnitedKingdom

cal = UnitedKingdom()

# collect all UK bank holiday dates across the data range
holiday_dates = set()
for year in range(weekly['week_start'].dt.year.min(), weekly['week_start'].dt.year.max() + 2):
    for date, _ in cal.holidays(year):
        holiday_dates.add(pd.Timestamp(date))

def week_contains_holiday(week_start):
    week_end = week_start + pd.Timedelta(days=6)
    return any(week_start <= h <= week_end for h in holiday_dates)

unique_weeks = weekly['week_start'].unique()
holiday_map = {w: int(week_contains_holiday(w)) for w in unique_weeks}

weekly['is_holiday_week'] = weekly['week_start'].map(holiday_map)
weekly['is_christmas_week'] = weekly['week_start'].dt.isocalendar().week.isin([51, 52]).astype(int)

print(f'Holiday weeks: {weekly["is_holiday_week"].sum() // len(all_products)}')
print(f'Christmas weeks: {weekly["is_christmas_week"].sum() // len(all_products)}')

Holiday weeks: 11
Christmas weeks: 3


## 6. Add calendar features

In [7]:
iso = weekly['week_start'].dt.isocalendar()
weekly['week_of_year'] = iso.week.astype(int)
weekly['year'] = iso.year.astype(int)

# cyclical encoding of week-of-year for annual seasonality
weekly['week_sin'] = np.sin(2 * np.pi * weekly['week_of_year'] / 52)
weekly['week_cos'] = np.cos(2 * np.pi * weekly['week_of_year'] / 52)

weekly.head()

,product_family_name,week_start,weekly_sales,is_holiday_week,is_christmas_week,week_of_year,year,week_sin,week_cos
0,*Boombox Ipod Classic,2009-12-01,0.0,0,0,49,2009,-3.546049e-01,0.935016
1,*Boombox Ipod Classic,2009-12-08,0.0,0,0,50,2009,-2.393157e-01,0.970942
2,*Boombox Ipod Classic,2009-12-15,0.0,0,1,51,2009,-1.205367e-01,0.992709
3,*Boombox Ipod Classic,2009-12-22,0.0,1,1,52,2009,6.432491e-16,1.000000
4,*Boombox Ipod Classic,2009-12-29,0.0,1,0,53,2009,1.205367e-01,0.992709


## 7. Chronological train / val / test split (70 / 15 / 15)

In [8]:
n = len(all_weeks)
train_end = all_weeks[int(n * 0.70) - 1]
val_end   = all_weeks[int(n * 0.85) - 1]

print(f'Total weeks : {n}')
print(f'Train end   : {train_end}  ({int(n*0.70)} weeks)')
print(f'Val end     : {val_end}   ({int(n*0.15)} weeks)')
print(f'Test        : {int(n*0.15)} weeks')

train = weekly[weekly['week_start'] <= train_end].copy()
val   = weekly[(weekly['week_start'] > train_end) & (weekly['week_start'] <= val_end)].copy()
test  = weekly[weekly['week_start'] > val_end].copy()

print(f'\nTrain rows: {len(train):,}')
print(f'Val rows  : {len(val):,}')
print(f'Test rows : {len(test):,}')

Total weeks : 105
Train end   : 2011-04-26 00:00:00  (73 weeks)
Val end     : 2011-08-16 00:00:00   (15 weeks)
Test        : 15 weeks

Train rows: 187,902
Val rows  : 41,184
Test rows : 41,184


## 8. Save

## 9. Join cluster assignments (from prior team's K-Means)

In [9]:
cl = pd.read_parquet(OUT / 'clustering' / 'clusters_3models.parquet')[['cluster_kmeans']].rename(
    columns={'cluster_kmeans': 'cluster'}
)

for split_df, fname in [(train, 'weekly_train'), (val, 'weekly_val'), (test, 'weekly_test')]:
    out_df = split_df.join(cl, on='product_family_name', how='left')
    out_df.to_parquet(OUT / f'{fname}_clustered.parquet', index=False)
    n_clustered = out_df['cluster'].notna().sum() // out_df['cluster'].notna().groupby(
        out_df['product_family_name']
    ).ngroup().nunique() if False else out_df['cluster'].notna().sum()
    print(f'{fname}_clustered: {len(out_df):,} rows | cluster dist: '
          f'{out_df.groupby("cluster")["product_family_name"].nunique().to_dict()}')

print('Done.')

weekly_train_clustered: 187,902 rows | cluster dist: {0.0: 515, 1.0: 473, 2.0: 10, 3.0: 1017, 4.0: 8}
weekly_val_clustered: 41,184 rows | cluster dist: {0.0: 515, 1.0: 473, 2.0: 10, 3.0: 1017, 4.0: 8}


weekly_test_clustered: 41,184 rows | cluster dist: {0.0: 515, 1.0: 473, 2.0: 10, 3.0: 1017, 4.0: 8}
Done.


In [10]:
train.to_parquet(OUT / 'weekly_train.parquet', index=False)
val.to_parquet(OUT / 'weekly_val.parquet', index=False)
test.to_parquet(OUT / 'weekly_test.parquet', index=False)

print('Saved weekly_train / val / test parquets')
print('Columns:', train.columns.tolist())

Saved weekly_train / val / test parquets
Columns: ['product_family_name', 'week_start', 'weekly_sales', 'is_holiday_week', 'is_christmas_week', 'week_of_year', 'year', 'week_sin', 'week_cos']
